<a href="https://colab.research.google.com/github/VukasinA/ML_projekti/blob/main/Domaci2GAVA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Domaci zadatak 2

In [ ]:
import random
import sys
from collections import defaultdict
from google.colab import files
import time

class GeneticMST:
    def __init__(self, graph, population_size=100, generations=300,
                 crossover_rate=0.85, mutation_rate=0.15, elitism=3):
        self.graph = graph
        self.num_nodes = len(graph)
        self.population_size = population_size
        self.generations = generations
        self.crossover_rate = crossover_rate
        self.mutation_rate = mutation_rate
        self.elitism = elitism

        # Сви могући ивице у графу
        self.all_edges = []
        for i in range(self.num_nodes):
            for j in range(i+1, self.num_nodes):
                self.all_edges.append((i, j, self.graph[i][j]))

    def initialize_population(self):
        population = []
        for _ in range(self.population_size):
            # Користимо Примов алгоритам за генерисање валидних стабала
            tree = self.generate_random_spanning_tree()
            population.append(tree)
        return population

    def generate_random_spanning_tree(self):
        nodes = set()
        edges = []

        # Изабери почетни чвор насумично
        start_node = random.randint(0, self.num_nodes - 1)
        nodes.add(start_node)

        while len(nodes) < self.num_nodes:
            # Нађи све ивице које повезују чворове у стаблу са осталим чворовима
            candidate_edges = []
            for node in nodes:
                for neighbor in range(self.num_nodes):
                    if neighbor not in nodes and neighbor != node:
                        candidate_edges.append((node, neighbor, self.graph[node][neighbor]))

            # Изабери насумичну ивицу и додај је у стабло
            if candidate_edges:
                edge = random.choice(candidate_edges)
                edges.append((edge[0], edge[1]))
                nodes.add(edge[1])

        return edges

    def calculate_fitness(self, individual):
        total_weight = sum(self.graph[u][v] for u, v in individual)
        return total_weight

    def selection(self, population):
        # Турнирска селекција са већом вероватноћом за боље индивидуе
        tournament_size = 5
        tournament = random.sample(population, tournament_size)
        tournament.sort(key=lambda x: self.calculate_fitness(x))
        return tournament[0]

    def crossover(self, parent1, parent2):
        if random.random() > self.crossover_rate:
            return parent1, parent2

        # Креирамо унију грана родитеља
        union_edges = list(set(parent1) | set(parent2))

        # Користимо Крускалов алгоритам да би осигурали да је дете валидно стабло
        union_edges_with_weights = [(u, v, self.graph[u][v]) for u, v in union_edges]
        union_edges_with_weights.sort(key=lambda x: x[2])

        parent = defaultdict(int)

        def find(u):
            while parent[u] != u:
                parent[u] = parent[parent[u]]
                u = parent[u]
            return u

        def union(u, v):
            u_root = find(u)
            v_root = find(v)
            if u_root == v_root:
                return False
            parent[v_root] = u_root
            return True

        child_edges = []
        nodes_in_child = set()

        for u, v, w in union_edges_with_weights:
            if u not in parent:
                parent[u] = u
            if v not in parent:
                parent[v] = v

            if union(u, v):
                child_edges.append((u, v))
                nodes_in_child.add(u)
                nodes_in_child.add(v)
                if len(nodes_in_child) == self.num_nodes:
                    break

        return child_edges, parent2

    def mutation(self, individual):
        if random.random() > self.mutation_rate:
            return individual

        # Изабери случајну грану у стаблу да је уклонимо
        if len(individual) == 0:
            return individual

        edge_to_remove = random.choice(individual)
        new_individual = [edge for edge in individual if edge != edge_to_remove]

        # Нађи све могуће гране које могу да повежу стабло
        nodes_in_tree = set()
        for u, v in new_individual:
            nodes_in_tree.add(u)
            nodes_in_tree.add(v)

        candidate_edges = []
        for u in nodes_in_tree:
            for v in range(self.num_nodes):
                if v not in nodes_in_tree and (u, v) not in new_individual and (v, u) not in new_individual:
                    candidate_edges.append((u, v))

        if candidate_edges:
            new_edge = random.choice(candidate_edges)
            new_individual.append(new_edge)

        return new_individual

    def evolve(self):
        population = self.initialize_population()

        for generation in range(self.generations):
            # Сортирај популацију по фитнесу
            population.sort(key=lambda x: self.calculate_fitness(x))

            # Елитизам - задржи најбоље индивидуе
            new_population = population[:self.elitism]

            while len(new_population) < self.population_size:
                parent1 = self.selection(population)
                parent2 = self.selection(population)

                child1, child2 = self.crossover(parent1, parent2)

                child1 = self.mutation(child1)
                child2 = self.mutation(child2)

                new_population.append(child1)
                if len(new_population) < self.population_size:
                    new_population.append(child2)

            population = new_population

        # Врати најбоље решење
        population.sort(key=lambda x: self.calculate_fitness(x))
        return population[0]

    def run(self, num_runs=3):
        best_solution = None
        best_fitness = float('inf')
        total_fitness = 0
        all_solutions = []

        for _ in range(num_runs):
            solution = self.evolve()
            fitness = self.calculate_fitness(solution)
            total_fitness += fitness
            all_solutions.append((solution, fitness))

            if fitness < best_fitness:
                best_fitness = fitness
                best_solution = solution

        average_fitness = total_fitness / num_runs
        return best_solution, best_fitness, average_fitness, all_solutions

def read_graph_from_file(file_content):
    lines = [line.strip() for line in file_content.split('\n') if line.strip()]

    n = int(lines[0])
    graph = [[0.0] * n for _ in range(n)]

    for i in range(n-1):
        weights = []
        parts = lines[i+1].split()
        for part in parts:
            try:
                # Обрада и целобројних и децималних вредности
                if '.' in part:
                    weights.append(float(part))
                else:
                    weights.append(int(part))
            except ValueError:
                continue

        for j in range(len(weights)):
            graph[i][i+1+j] = weights[j]
            graph[i+1+j][i] = weights[j]

    return graph

def format_solution(solution):
    # Сортирај гране
    sorted_edges = sorted([(min(u, v), max(u, v)) for u, v in solution])
    return " ".join([f"({u} {v})" for u, v in sorted_edges])

def main():
    print("Генетски алгоритам за MST проблем")
    print("--------------------------------")

    # Опција за учитавање фајла
    print("\n1. Учитајте фајл са свог рачунара")
    uploaded = files.upload()
    if not uploaded:
      print("Није одабран ниједан фајл! Програм се прекида.")
      return

    file_name = next(iter(uploaded))
    file_content = uploaded[file_name].decode('utf-8')
    graph = read_graph_from_file(file_content)

    # Прилагођавање параметара на основу величине графа
    n = len(graph)
    population_size = min(200, max(50, n * 2))
    generations = min(500, max(100, n*2))

    print("\nПараметри алгоритма:")
    print(f"- Величина популације: {population_size}")
    print(f"- Број генерација: {generations}")
    print(f"- Вероватноћа укрштања: 0.85")
    print(f"- Вероватноћа мутације: 0.15")
    print(f"- Елитизам: 3")

    print("\nПокрећем алгоритам...")
    start_time = time.time()

    genetic_mst = GeneticMST(graph,
                           population_size=population_size,
                           generations=generations,
                           crossover_rate=0.85,
                           mutation_rate=0.15)

    best_solution, best_fitness, avg_fitness, all_solutions = genetic_mst.run()

    end_time = time.time()
    execution_time = end_time - start_time

    print("\nРезултати:")
    print("---------")
    print(f"Најбоља пронађена тежина: {best_fitness:.2f}".rstrip('0').rstrip('.') if '.' in f"{best_fitness:.2f}" else f"{best_fitness}")
    print("Гране које чине MST:")
    print(format_solution(best_solution))

    print("\nДетаљни резултати:")
    print(f"- Просечна тежина (3 покрета): {avg_fitness:.2f}")
    print(f"- Време извршавања: {execution_time:.2f} секунди")

    print("\nСва решења из покретања:")
    for i, (sol, fit) in enumerate(all_solutions, 1):
        print(f"Покретање {i}: Тежина = {fit:.2f}, Гране = {format_solution(sol)}")

if __name__ == "__main__":
    main()

Генетски алгоритам за MST проблем
--------------------------------

1. Учитајте фајл са свог рачунара


Saving SYM304_.txt to SYM304_ (1).txt

Параметри алгоритма:
- Величина популације: 60
- Број генерација: 100
- Вероватноћа укрштања: 0.85
- Вероватноћа мутације: 0.15
- Елитизам: 3

Покрећем алгоритам...

Резултати:
---------
Најбоља пронађена тежина: 1023
Гране које чине MST:
(0 1) (1 2) (1 12) (3 8) (4 25) (5 7) (6 11) (7 18) (8 28) (9 16) (9 24) (13 19) (14 25) (15 27) (17 20) (17 27) (17 29) (21 22)

Детаљни резултати:
- Просечна тежина (3 покрета): 1177.67
- Време извршавања: 0.94 секунди

Сва решења из покретања:
Покретање 1: Тежина = 1023.00, Гране = (0 1) (1 2) (1 12) (3 8) (4 25) (5 7) (6 11) (7 18) (8 28) (9 16) (9 24) (13 19) (14 25) (15 27) (17 20) (17 27) (17 29) (21 22)
Покретање 2: Тежина = 1343.00, Гране = (0 1) (1 12) (2 4) (2 5) (3 8) (3 24) (4 25) (6 11) (6 20) (7 9) (7 18) (9 16) (9 24) (13 15) (13 19) (14 23) (14 25) (15 21) (17 20) (17 27) (17 29) (20 26) (22 25) (24 27)
Покретање 3: Тежина = 1167.00, Гране = (0 1) (0 20) (1 2) (1 12) (3 8) (3 24) (4 25) (5 7) (6 